# 🌌 IA-B (RECEPTOR): Asimilación de HuggingFace + Multimedia

Este cuaderno asimila el archivo .pmtp generado por la IA-A.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
BUS_DIR = '/content/drive/MyDrive/POLYDIM_BUS/DEMO_PMTP/bus'
RECEIVE_DIR = '/content/drive/MyDrive/POLYDIM_BUS/DEMO_PMTP/receive'
os.makedirs(RECEIVE_DIR, exist_ok=True)

In [ ]:
import time
import numpy as np
import jax
import jax.numpy as jnp
import struct

pmtp_path = os.path.join(BUS_DIR, 'pmtp_hf_transfer.pmtp')
if not os.path.exists(pmtp_path):
    print('❌ ERROR: Tensor no encontrado.')
else:
    print('Asimilando PMTP...')
    file_size_bytes = os.path.getsize(pmtp_path)
    N = file_size_bytes // (4 * 1024)
    
    fp = np.memmap(pmtp_path, dtype=np.float32, mode='r', shape=(N, 1024))
    Z = jnp.array(fp)
    jax.block_until_ready(Z)
    
    raw_bytes = np.asarray(Z).astype(np.int8).tobytes()
    header = raw_bytes[:264]
    name_end = header[:256].find(b'\x00')
    if name_end == -1: name_end = 256
    orig_name = header[:name_end].decode('utf-8')
    orig_size = struct.unpack('<Q', header[256:264])[0]
    
    payload = raw_bytes[264:264+orig_size]
    out_path = os.path.join(RECEIVE_DIR, orig_name)
    with open(out_path, 'wb') as f:
        f.write(payload)
        
    print(f'✅ Archivo {orig_name} ({orig_size} bytes) reconstruido.')
    print(f'Contenido Asimilado:\n{payload.decode("utf-8")}')